In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.dominick import DominickDataLoader

import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

from scipy.stats import ttest_rel, wilcoxon

In [ ]:
# ============================================
# 1. Configuración
# ============================================

NN_KFOLD_METRICS_PATH = "nn_kfold_metrics_raw.csv"
NN_BOOT_OWN_SUMMARY_PATH = "nn_bootstrap_own_summary.csv"
NN_BOOT_RAW_PATH = "nn_bootstrap_elasticities_raw.csv"

BENCH_KFOLD_RAW_PATH = "benchmark_kfold_raw.csv"                
BENCH_BOOT_SUMMARY_PATH = "benchmark_elasticities_bootstrap.csv"
BENCH_BOOT_RAW_PATH = "benchmark_bootstrap_raw.csv"            

KEY_COLS = ["store_code", "upc_code"]
ABS_ERROR_THRESHOLD = 0.5

In [ ]:
# ============================================
# 2. Carga de datos
# ============================================

loader = DominickDataLoader()

def safe_load(path):
    try:
        df = loader.load(path)
        print(f"[OK] {path}: {df.shape}")
        return df
    except Exception as e:
        print(f"[WARN] No se pudo cargar {path}: {e}")
        return None

nn_kfold = safe_load(NN_KFOLD_METRICS_PATH)
nn_boot_own = safe_load(NN_BOOT_OWN_SUMMARY_PATH)
nn_boot_raw = safe_load(NN_BOOT_RAW_PATH)

bench_kfold = safe_load(BENCH_KFOLD_RAW_PATH)
bench_boot = safe_load(BENCH_BOOT_SUMMARY_PATH)
bench_boot_raw = safe_load(BENCH_BOOT_RAW_PATH)

In [ ]:
# ============================================
# 3. BLOQUE 1 — Generalización
# ============================================

if nn_kfold is None or bench_kfold is None:
    raise ValueError(
        "Faltan nn_kfold_metrics_raw.csv o benchmark_kfold_raw.csv. "
        "Guarda los raw de ambos notebooks antes de correr este análisis."
    )

required_nn_cols = {"fold", "seed", "mae_val", "rmse_val", "r2_val"}
required_bench_cols = {"fold", "mae_val", "rmse_val", "r2_val"}

missing_nn = required_nn_cols - set(nn_kfold.columns)
missing_bench = required_bench_cols - set(bench_kfold.columns)

if missing_nn:
    raise ValueError(f"Faltan columnas en nn_kfold: {missing_nn}")
if missing_bench:
    raise ValueError(f"Faltan columnas en benchmark_kfold: {missing_bench}")

# NN: agregamos por fold promediando seeds
nn_by_fold = (
    nn_kfold
    .groupby("fold", as_index=False)
    .agg(
        mae_val_nn=("mae_val", "mean"),
        rmse_val_nn=("rmse_val", "mean"),
        r2_val_nn=("r2_val", "mean"),
        r2_seed_std_nn=("r2_val", "std"),
        mae_seed_std_nn=("mae_val", "std"),
        rmse_seed_std_nn=("rmse_val", "std"),
        n_seeds=("seed", "nunique"),
    )
)

# Benchmark: agregamos por fold promediando series
bench_by_fold = (
    bench_kfold
    .groupby("fold", as_index=False)
    .agg(
        mae_val_benchmark=("mae_val", "mean"),
        rmse_val_benchmark=("rmse_val", "mean"),
        r2_val_benchmark=("r2_val", "mean"),
        n_series=("r2_val", "size"),
    )
)

gen = nn_by_fold.merge(bench_by_fold, on="fold", how="inner")
print("Folds comparables:", len(gen))
display(gen)

In [ ]:
# ============================================
# 4. BLOQUE 1 — Resumen de generalización
# ============================================

gen["delta_r2"] = gen["r2_val_nn"] - gen["r2_val_benchmark"]
gen["delta_mae"] = gen["mae_val_nn"] - gen["mae_val_benchmark"]
gen["delta_rmse"] = gen["rmse_val_nn"] - gen["rmse_val_benchmark"]

summary = {
    "n_folds": len(gen),
    "mean_r2_nn": gen["r2_val_nn"].mean(),
    "mean_r2_benchmark": gen["r2_val_benchmark"].mean(),
    "mean_delta_r2": gen["delta_r2"].mean(),
    "median_delta_r2": gen["delta_r2"].median(),
    "nn_wins_r2_rate": (gen["delta_r2"] > 0).mean(),
    "mean_mae_nn": gen["mae_val_nn"].mean(),
    "mean_mae_benchmark": gen["mae_val_benchmark"].mean(),
    "mean_delta_mae": gen["delta_mae"].mean(),
    "nn_wins_mae_rate": (gen["delta_mae"] < 0).mean(),
    "mean_rmse_nn": gen["rmse_val_nn"].mean(),
    "mean_rmse_benchmark": gen["rmse_val_benchmark"].mean(),
    "mean_delta_rmse": gen["delta_rmse"].mean(),
    "nn_wins_rmse_rate": (gen["delta_rmse"] < 0).mean(),
    "mean_nn_seed_std_r2": gen["r2_seed_std_nn"].mean(),
}

if ttest_rel is not None and len(gen) >= 2:
    summary["ttest_delta_r2_pvalue"] = ttest_rel(gen["r2_val_nn"], gen["r2_val_benchmark"]).pvalue
    summary["ttest_delta_mae_pvalue"] = ttest_rel(gen["mae_val_nn"], gen["mae_val_benchmark"]).pvalue
    summary["ttest_delta_rmse_pvalue"] = ttest_rel(gen["rmse_val_nn"], gen["rmse_val_benchmark"]).pvalue

if wilcoxon is not None and len(gen) >= 2:
    try:
        summary["wilcoxon_delta_r2_pvalue"] = wilcoxon(gen["delta_r2"]).pvalue
    except Exception:
        summary["wilcoxon_delta_r2_pvalue"] = np.nan
    try:
        summary["wilcoxon_delta_mae_pvalue"] = wilcoxon(gen["delta_mae"]).pvalue
    except Exception:
        summary["wilcoxon_delta_mae_pvalue"] = np.nan
    try:
        summary["wilcoxon_delta_rmse_pvalue"] = wilcoxon(gen["delta_rmse"]).pvalue
    except Exception:
        summary["wilcoxon_delta_rmse_pvalue"] = np.nan

generalization_summary = pd.DataFrame.from_dict(summary, orient="index", columns=["value"])
display(generalization_summary)

In [ ]:
# ============================================
# 5. BLOQUE 1 — Estabilidad de la NN entre seeds
# ============================================

nn_seed_stability = (
    nn_kfold
    .groupby("fold", as_index=False)
    .agg(
        n_seeds=("seed", "nunique"),
        r2_mean=("r2_val", "mean"),
        r2_std=("r2_val", "std"),
        mae_mean=("mae_val", "mean"),
        mae_std=("mae_val", "std"),
        rmse_mean=("rmse_val", "mean"),
        rmse_std=("rmse_val", "std"),
    )
)

print("Estabilidad entre seeds por fold")
display(nn_seed_stability)

In [ ]:
# ============================================
# 6. BLOQUE 2 — Elasticidad mejor calculada / más estable
# ============================================

if nn_boot_own is None or bench_boot is None:
    raise ValueError(
        "Faltan nn_bootstrap_own_summary.csv o benchmark_elasticities_bootstrap.csv"
    )

elasticity_cmp = nn_boot_own.merge(
    bench_boot,
    on=KEY_COLS,
    how="inner",
    suffixes=("_nn", "_benchmark")
)

print("Series comparables para elasticidad:", len(elasticity_cmp))
display(elasticity_cmp.head())

In [ ]:
# ============================================
# 7. BLOQUE 2 — Resumen de estabilidad de elasticidad
# ============================================

elasticity_cmp["elasticity_gap"] = (
    elasticity_cmp["elasticity_mean_nn"] - elasticity_cmp["elasticity_mean_benchmark"]
)
elasticity_cmp["abs_gap"] = elasticity_cmp["elasticity_gap"].abs()

elasticity_cmp["ci_width_nn"] = (
    elasticity_cmp["elasticity_ci_high_nn"] - elasticity_cmp["elasticity_ci_low_nn"]
)
elasticity_cmp["ci_width_benchmark"] = (
    elasticity_cmp["elasticity_ci_high_benchmark"] - elasticity_cmp["elasticity_ci_low_benchmark"]
)

elasticity_cmp["same_sign"] = (
    np.sign(elasticity_cmp["elasticity_mean_nn"]) ==
    np.sign(elasticity_cmp["elasticity_mean_benchmark"])
)

elasticity_cmp["nn_more_stable_std"] = (
    elasticity_cmp["elasticity_std_nn"] < elasticity_cmp["elasticity_std_benchmark"]
)
elasticity_cmp["nn_narrower_ci"] = (
    elasticity_cmp["ci_width_nn"] < elasticity_cmp["ci_width_benchmark"]
)

elasticity_stability_summary = pd.DataFrame({
    "metric": [
        "n_common_series",
        "mean_abs_gap",
        "median_abs_gap",
        "same_sign_rate",
        "nn_more_stable_std_rate",
        "nn_narrower_ci_rate",
        "mean_std_nn",
        "mean_std_benchmark",
        "mean_ci_width_nn",
        "mean_ci_width_benchmark",
    ],
    "value": [
        len(elasticity_cmp),
        elasticity_cmp["abs_gap"].mean(),
        elasticity_cmp["abs_gap"].median(),
        elasticity_cmp["same_sign"].mean(),
        elasticity_cmp["nn_more_stable_std"].mean(),
        elasticity_cmp["nn_narrower_ci"].mean(),
        elasticity_cmp["elasticity_std_nn"].mean(),
        elasticity_cmp["elasticity_std_benchmark"].mean(),
        elasticity_cmp["ci_width_nn"].mean(),
        elasticity_cmp["ci_width_benchmark"].mean(),
    ]
})

display(elasticity_stability_summary)

In [ ]:
# ============================================
# 8. BLOQUE 2 — Estabilidad de signo en bootstrap
# ============================================

sign_stability = None

if nn_boot_raw is not None and bench_boot_raw is not None:
    nn_boot_own_raw = nn_boot_raw[nn_boot_raw["tipo"] == "own"].copy()

    nn_sign = (
        nn_boot_own_raw
        .groupby(KEY_COLS, as_index=False)
        .agg(
            nn_negative_rate=("E", lambda x: (x < 0).mean()),
            nn_boot_n=("E", "size"),
        )
    )

    bench_sign = (
        bench_boot_raw
        .groupby(KEY_COLS, as_index=False)
        .agg(
            benchmark_negative_rate=("elasticity", lambda x: (x < 0).mean()),
            benchmark_boot_n=("elasticity", "size"),
        )
    )

    sign_stability = nn_sign.merge(bench_sign, on=KEY_COLS, how="inner")
    sign_stability["nn_more_negative_stable"] = (
        sign_stability["nn_negative_rate"] > sign_stability["benchmark_negative_rate"]
    )

    print("Sign stability bootstrap:")
    display(sign_stability.head())

else:
    print("No hay raw bootstrap completos para comparar estabilidad de signo.")

In [ ]:
# ============================================
# 9. BLOQUE 1 + 2 — Visualizaciones
# ============================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Delta R2 por fold
axes[0].bar(gen["fold"].astype(str), gen["delta_r2"])
axes[0].axhline(0, linestyle="--")
axes[0].set_title("Delta R² por fold (NN - Benchmark)")
axes[0].set_xlabel("Fold")
axes[0].set_ylabel("Delta R²")
axes[0].grid(True, alpha=0.3)

# Ancho IC NN vs Benchmark
axes[1].scatter(elasticity_cmp["ci_width_benchmark"], elasticity_cmp["ci_width_nn"], alpha=0.7)
min_val = min(elasticity_cmp["ci_width_benchmark"].min(), elasticity_cmp["ci_width_nn"].min())
max_val = max(elasticity_cmp["ci_width_benchmark"].max(), elasticity_cmp["ci_width_nn"].max())
axes[1].plot([min_val, max_val], [min_val, max_val], linestyle="--")
axes[1].set_xlabel("CI width Benchmark")
axes[1].set_ylabel("CI width NN")
axes[1].set_title("Anchura de intervalos: NN vs Benchmark")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# 11. BLOQUE 3 — ¿Qué puedo concluir de verdad?
# ============================================

generalization_ok = (
    generalization_summary.loc["mean_delta_r2", "value"] > 0
    and generalization_summary.loc["mean_delta_mae", "value"] < 0
    and generalization_summary.loc["mean_delta_rmse", "value"] < 0
)

elasticity_more_stable = (
    elasticity_stability_summary
    .set_index("metric")
    .loc["nn_more_stable_std_rate", "value"] > 0.5
)

intervals_narrower = (
    elasticity_stability_summary
    .set_index("metric")
    .loc["nn_narrower_ci_rate", "value"] > 0.5
)

conclusion_table = pd.DataFrame({
    "claim": [
        "La NN generaliza mejor que OLS",
        "La elasticidad de la NN está mejor calculada / es más estable",
        "La elasticidad de la NN es más precisa",
    ],
    "status": [
        "sí" if generalization_ok else "no / débil",
        "parcialmente sí" if (generalization_ok and elasticity_more_stable) else "no / débil",
        "no todavía",
    ],
    "reason": [
        "Se sostiene si mejora R² y reduce MAE/RMSE en folds temporales.",
        "Se sostiene parcialmente si además la elasticidad NN tiene menor std / IC en bootstrap.",
        "No basta con intervalos más estrechos. Falta calibración o verdad externa.",
    ]
})

display(conclusion_table)